# Phase 3 — SFT warm-start of AV + AR on Colab T4

**Prereqs**
1. Upload `guppylm-nla-bundle.tar.gz` (~114 MB) to your Google Drive `/MyDrive/`.
2. Runtime → Change runtime type → T4 GPU.

**Outputs** (saved back to Drive `/MyDrive/guppylm-nla-out/`)
- `history_text.json`, `history_lens.json` — per-variant training curves + final FVE
- `MANIFEST_phase3.json` — provenance + decision banner
- `checkpoints/phase3/{av,ar}_{text,lens}/step_*.pt` — last-step checkpoints

## 1. Setup (pip install + mount Drive + untar)

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q 'transformers>=4.43' 'accelerate>=0.30' 'peft>=0.12' 'datasets>=2.20' \
    'bitsandbytes>=0.43' 'numpy>=1.24' 'tqdm>=4.65' 'tokenizers>=0.19' \
    'huggingface_hub>=0.20' 'pyarrow>=14' 'pytest>=7.4' 'pytest-asyncio>=0.21' \
    'openai>=1.40'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
BUNDLE = '/content/drive/MyDrive/guppylm-nla-bundle.tar.gz'
WORK = '/content/guppylm-infer'
os.makedirs(WORK, exist_ok=True)
!tar xzf {BUNDLE} -C {WORK}
%cd {WORK}
!ls -la data/ checkpoints/

## 2. Sanity tests (no GPU; ~10s)

In [ ]:
!python -m pytest tests/test_av_ar.py -v

## 3. Smoke run (~3 min) — verifies the full pipeline on the actual GPU

In [ ]:
!python scripts/03_warmstart.py --variant text \
    --max-steps 50 --min-steps 0 --time-budget-min 5 --batch 2 \
    --history-root data/smoke --ckpt-root checkpoints/phase3_smoke \
    --manifest data/MANIFEST_phase3_smoke.json

In [ ]:
import json
smoke = json.loads(open('data/smoke/history_text.json').read())
print('smoke FVE:', smoke['final_fve'])
print('smoke MSE:', smoke['final_mse'])
print('AV stop:', smoke['av']['stop_reason'], 'steps:', smoke['av']['final_step'])
print('AR stop:', smoke['ar']['stop_reason'], 'steps:', smoke['ar']['final_step'])
if smoke['samples']:
    print('\nfirst sample:'); print(smoke['samples'][0])

## 4. Full text variant (~60–180 min, convergence- or budget-bounded)

In [ ]:
!python scripts/03_warmstart.py --variant text --time-budget-min 180

## 5. Full lens variant (~60–180 min)

In [ ]:
!python scripts/03_warmstart.py --variant lens --time-budget-min 180

## 6. Plot FVE curves and apply decision rule

In [ ]:
import json
import matplotlib.pyplot as plt
ht = json.load(open('data/history_text.json'))
hl = json.load(open('data/history_lens.json'))

def fve_curve(h):
    return [(e['step'], e['fve']) for e in h['ar']['history'] if 'fve' in e]

fig, ax = plt.subplots(figsize=(10, 5))
for h, label, color in [(ht, 'text', 'C0'), (hl, 'lens', 'C1')]:
    pts = fve_curve(h)
    if pts:
        xs, ys = zip(*pts)
        ax.plot(xs, ys, color=color, label=f'{label} (final={h["final_fve"]:.3f})', marker='o')
ax.axhline(0.10, color='red', linestyle='--', alpha=0.5, label='hard-fail floor (0.10)')
ax.axhline(0.25, color='green', linestyle='--', alpha=0.5, label='Phase 4 threshold (0.25)')
ax.set_xlabel('AR training step'); ax.set_ylabel('FVE')
ax.set_title('Joint AV→AR FVE on held-out 500')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('data/fve_curves.png', dpi=120)
plt.show()

In [ ]:
!cat data/MANIFEST_phase3.json

## 7. Save artifacts back to Drive

In [ ]:
import os, shutil
OUT = '/content/drive/MyDrive/guppylm-nla-out'
os.makedirs(OUT, exist_ok=True)
for f in ['history_text.json', 'history_lens.json', 'MANIFEST_phase3.json',
          'fve_curves.png', 'splits.json']:
    src = f'data/{f}'
    if os.path.exists(src):
        shutil.copy2(src, OUT)
        print('saved', f)
# Tarball checkpoints (large; optional).
!tar czf /tmp/phase3_ckpts.tar.gz checkpoints/phase3/
shutil.copy2('/tmp/phase3_ckpts.tar.gz', OUT)
print('saved phase3_ckpts.tar.gz')
!ls -la {OUT}